# Ramp-up — Image data fundamentals

**Goal.** Build the minimum image-data fluency the workshop assumes:
- what a digital image is at the data-structure level
- shape, dtype, channels, and dimensions
- common file formats in microscopy
- displaying images with appropriate contrast
- the relationship between pixel coordinates and physical units

**Time.** ~45 minutes. Hands-on the whole way through.

## A digital image is just an array of numbers

In [ ]:
%pip install --quiet numpy matplotlib scikit-image tifffile
import numpy as np
import matplotlib.pyplot as plt
from skimage import io as skio

## Make and display a simple image

In [ ]:
# A 2D array — grayscale image
img = np.zeros((100, 100), dtype=np.uint8)
img[20:80, 20:80] = 200   # bright square in the middle
img[40:60, 40:60] = 100   # darker square inside

print("Shape :", img.shape)
print("Dtype :", img.dtype)
print("Min   :", img.min(), "  Max:", img.max())

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(img, cmap='gray', vmin=0, vmax=255)
ax.set_title(f"Grayscale {img.shape}")
plt.tight_layout(); plt.show()

## Multi-channel images: shape conventions

In [ ]:
# A 3-channel RGB-like image: shape (H, W, C)
rgb = np.zeros((100, 100, 3), dtype=np.uint8)
rgb[..., 0] = 200    # full red
rgb[40:60, 40:60, 1] = 200    # green square in the middle

print("Shape :", rgb.shape)
print("Dtype :", rgb.dtype)
print("Per-channel means:", rgb.mean(axis=(0,1)))

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(rgb)
ax.set_title("RGB image (H, W, 3)")
plt.tight_layout(); plt.show()

**Convention warning.** Microscopy data sometimes uses (C, H, W) instead of (H, W, C). Tools differ. Always check the shape and use the cell below to detect channels-first vs channels-last format.

In [ ]:
def channels_axis(img):
    """Heuristic: the smallest axis (size <= 4) is typically the channel axis."""
    if img.ndim < 3: return None
    sizes = list(img.shape)
    smallest = min(sizes)
    if smallest <= 4:
        return sizes.index(smallest)
    return None

print("Channels-last RGB (H, W, C):", rgb.shape, "→ channels axis:", channels_axis(rgb))

channels_first = np.transpose(rgb, (2, 0, 1))  # (C, H, W)
print("Channels-first RGB (C, H, W):", channels_first.shape, "→ channels axis:", channels_axis(channels_first))

## Bit depth

In [ ]:
# 8-bit: values 0-255
img_8bit = np.linspace(0, 255, 256).astype(np.uint8).reshape(16, 16)
print("8-bit  range:", img_8bit.min(), img_8bit.max())

# 16-bit: values 0-65535 (more dynamic range, common in microscopy)
img_16bit = np.linspace(0, 65535, 256).astype(np.uint16).reshape(16, 16)
print("16-bit range:", img_16bit.min(), img_16bit.max())

# Float: values typically 0.0-1.0 (after normalization)
img_float = np.linspace(0, 1, 256).reshape(16, 16)
print("Float range:", img_float.min().round(3), img_float.max().round(3))

**Why bit depth matters.** Most microscopes capture 12-bit or 16-bit raw data. Many tools convert to 8-bit for display, which discards 4-8 bits of dynamic range. For *quantitative* analysis you usually want to keep the original bit depth as long as possible.

## Displaying with appropriate contrast

In [ ]:
# A 16-bit image where most pixels are dark, with a few bright spots
img16 = np.zeros((100, 100), dtype=np.uint16)
img16[40:60, 40:60] = 50000   # bright square
img16 = img16 + np.random.randint(0, 200, img16.shape).astype(np.uint16)

# Default display: contrast is too low because matplotlib auto-scales the bright pixels
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(img16, cmap='gray'); axes[0].set_title("Default (no vmin/vmax)")

# Better: percentile-based contrast
p1, p99 = np.percentile(img16, [1, 99])
axes[1].imshow(img16, cmap='gray', vmin=p1, vmax=p99); axes[1].set_title(f"vmin={int(p1)}, vmax={int(p99)}")

# Best for quantitative: full data range, with a colorbar
axes[2].imshow(img16, cmap='viridis')
plt.colorbar(axes[2].images[0], ax=axes[2])
axes[2].set_title("With colorbar (quantitative)")
for a in axes: a.axis('off')
plt.tight_layout(); plt.show()

## File formats — TIFF and beyond

In [ ]:
import tifffile

# Save a TIFF and read it back
tifffile.imwrite("test.tif", img)
loaded = tifffile.imread("test.tif")
print("Saved and reloaded TIFF:", loaded.shape, loaded.dtype, "→ matches:", np.array_equal(img, loaded))

# OME-TIFF, NDPI, SVS, CZI, LIF, ND2, etc. are all supported by Bio-Formats
# and tools like AICSImageIO. For the workshop, TIFF and PNG cover most cases.

## Calibration: pixels and physical units

In [ ]:
# Microscopy images have a pixel size — usually in micrometers per pixel
pixel_size_um = 0.16    # e.g., 0.16 µm/pixel for a 60x objective at typical settings
image_shape = (1024, 1024)

field_of_view_um = (image_shape[0] * pixel_size_um, image_shape[1] * pixel_size_um)
print(f"Image  : {image_shape[0]} × {image_shape[1]} pixels")
print(f"FoV    : {field_of_view_um[0]:.1f} × {field_of_view_um[1]:.1f} µm")
print(f"Area   : {field_of_view_um[0] * field_of_view_um[1]:.1f} µm²")

**Key habit.** When you analyze image data quantitatively, the answer almost always needs to come out in physical units, not pixel units. The conversion is straightforward but easy to forget. Document the pixel size with every dataset.

## Closing reflection

You now have the image-data patterns the workshop assumes:

- digital images are arrays of numbers — `(H, W)`, `(H, W, C)`, or `(C, H, W)`
- bit depth matters; preserve it for quantitative analysis
- contrast adjustment is for display; original data is what you compute on
- pixel coordinates and physical units are not the same — keep track

Combined with the Python basics notebook, you have what tomorrow's workshop assumes. See you in the morning.